## SBA Data Cleaning

In [1]:
import pandas as pd
import numpy as np
import os

print(os.getcwd())

/Users/jack/Desktop/Fall 2026/CS4100/Final/CS4100-Final-Project/src/preprocessing


In [2]:
small_business_data = pd.read_csv("../../data/business_loans_2010_2025.csv")
small_business_data.columns

/var/folders/wj/tcpglhf11njbkfplcsjpk5800000gn/T/ipykernel_61683/4289294095.py:1: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  small_business_data = pd.read_csv("../../data/business_loans_2010_2025.csv")


Index(['asofdate', 'program', 'l2locid', 'borrname', 'borrstreet', 'borrcity',
       'borrstate', 'borrzip', 'cdc_name', 'cdc_street', 'cdc_city',
       'cdc_state', 'cdc_zip', 'thirdpartylender_name',
       'thirdpartylender_city', 'thirdpartylender_state', 'thirdpartydollars',
       'grossapproval', 'approvaldate', 'approvalfiscalyear',
       'firstdisbursementdate', 'processingmethod', 'deliverymethod',
       'subprogram', 'terminmonths', 'naicscode', 'naicsdescription',
       'franchisecode', 'franchisename', 'projectcounty', 'projectstate',
       'sbadistrictoffice', 'congressionaldistrict', 'businesstype',
       'businessage', 'loanstatus', 'paidinfulldate', 'chargeoffdate',
       'grosschargeoffamount', 'jobssupported', 'collateralind'],
      dtype='object')

In [3]:
small_business_data.loc[0]

asofdate                                                         12/31/2025
program                                                                 504
l2locid                                                            447141.0
borrname                                     Expressway 77 Hospitality, LLC
borrstreet                                         825 N. Expressway 77/83.
borrcity                                                        Brownsville
borrstate                                                                TX
borrzip                                                               78520
cdc_name                                                     LiftFund, Inc.
cdc_street                                            2014 S. Hackberry St.
cdc_city                                                        San Antonio
cdc_state                                                                TX
cdc_zip                                                             78210.0
thirdpartyle

In [4]:
small_business_data.isnull().sum()

asofdate                      0
program                       0
l2locid                       1
borrname                      0
borrstreet                    0
borrcity                      0
borrstate                     0
borrzip                       0
cdc_name                      1
cdc_street                    1
cdc_city                      1
cdc_state                     1
cdc_zip                       1
thirdpartylender_name       962
thirdpartylender_city       962
thirdpartylender_state      962
thirdpartydollars           949
grossapproval                 0
approvaldate                  0
approvalfiscalyear            0
firstdisbursementdate         0
processingmethod              0
deliverymethod            23531
subprogram                    0
terminmonths                  0
naicscode                   217
naicsdescription            188
franchisecode             21004
franchisename             21008
projectcounty                33
projectstate                  0
sbadistr

Keep only the columns needed for modelling

In [5]:
filtered_sb_data = small_business_data[[
    'asofdate', 'program', 'borrname', 'borrstate', 'borrzip',
    'thirdpartydollars', 'terminmonths',
    'grossapproval', 'approvaldate', 'approvalfiscalyear',
    'firstdisbursementdate', 'processingmethod', 'naicscode', 'collateralind',
    'businesstype', 'loanstatus', 'jobssupported',
    'grosschargeoffamount'
]].copy()

Convert approvaldate to an int

In [6]:
filtered_sb_data["approvalyear"] = pd.to_datetime(filtered_sb_data["approvaldate"]).dt.year
filtered_sb_data.drop("approvaldate", axis=1, inplace=True)

In [7]:
filtered_sb_data = filtered_sb_data.dropna(subset=['businesstype'])

### Data Exploration

In [8]:
print(filtered_sb_data.isnull().sum())

asofdate                   0
program                    0
borrname                   0
borrstate                  0
borrzip                    0
thirdpartydollars        949
terminmonths               0
grossapproval              0
approvalfiscalyear         0
firstdisbursementdate      0
processingmethod           0
naicscode                217
collateralind              0
businesstype               0
loanstatus                 0
jobssupported              0
grosschargeoffamount       0
approvalyear               0
dtype: int64


In [9]:
filtered_sb_data["loanstatus"].value_counts()

loanstatus
PIF       22605
CHGOFF      904
Name: count, dtype: int64

In [10]:
filtered_sb_data['grosschargeoffamount'].unique()

array([ 270002,   76845,  105644, 1096811,  326025, 1281538, 1012685,
        101789, 3582215,   39581,  318903,  218489,   89906,  401288,
       1904984,  615849,  839053,  122261,  293571, 1055660,  254041,
        252829,  377619,  130228,  123894,   95053, 1669039,   41989,
        413075,   79606,  482188,  599803,   94282, 1832362,  522777,
       1029544,  319765,  870399,  128789,  733291,  605073,  211883,
        146452,  472817,  964337,  459885,  157286,  464236,  700940,
        111145,   50826, 1472985,  993453,   68260,  428151,  669372,
        198778,  931537,  268951,  409755,  600151,  276678,  250926,
       1659451,  392372,   41780,   18492,  400635,  455892,  319382,
        119730,   40822,  417538, 2890743,  296382,  463116,  161622,
        726480,  169175, 2424739,  432465,   59100, 2529445,  227896,
        199874,  571160,   81557,  428985,  277566,  287566,  106264,
         92627,  123914,   26214,   57461,  946955,  213757,  177866,
        123217,  213

In [11]:
print(filtered_sb_data['businesstype'].value_counts())

businesstype
CORPORATION    21223
INDIVIDUAL      1771
PARTNERSHIP      515
Name: count, dtype: int64


Only PIF (paid in full) and CHGOFF (default) are meaningful outcomes for us, other loans are in progress

In [12]:
filtered_sb_data['loanstatus'].value_counts()

loanstatus
PIF       22605
CHGOFF      904
Name: count, dtype: int64

### Save Cleaned Data

In [13]:
filtered_sb_data.to_csv("../../data/cleaned_business_loans_2010_2025.csv", index=False)
print(filtered_sb_data.head())

     asofdate  program                            borrname borrstate  borrzip  \
0  12/31/2025      504      Expressway 77 Hospitality, LLC        TX    78520   
1  12/31/2025      504       LaSalle Service Station, Inc.        RI     2908   
2  12/31/2025      504  Robbie Male Design Solutions, Inc.        MA     1450   
3  12/31/2025      504              Arushi Enterprise Corp        MO    63042   
4  12/31/2025      504            Enfield Restaurants, LLC        KS    67654   

   thirdpartydollars  terminmonths  grossapproval  approvalfiscalyear  \
0         1640000.00           240         326000                2010   
1          182500.00           240         135000                2012   
2          147500.00           240         106000                2014   
3         2900000.00           240        2087000                2015   
4          552868.18           240         400000                2012   

  firstdisbursementdate            processingmethod  naicscode collaterali